In [ ]:
import copy
import gc
from importlib import reload
import os
import random
import sys

import numpy as np
from sklearn.preprocessing import StandardScaler
sys.path.append(os.path.abspath(os.path.join('..')))
import models

reload(models)

import torch
import pandas as pd
import mlflow
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from datetime import datetime

from models import HybridModel
from utils import mol_to_graph, MLFlowManager, train_hybrid_model, evaluate_hybrid_model
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') # type: ignore


import logging

logging.basicConfig(
    filename="debug_model.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    filemode="w"
)

logger = logging.getLogger(__name__)

def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_run(params, train_loader, test_loader, device, run_name):
    gc.collect()
    torch.cuda.empty_cache()

    model = HybridModel(
        num_node_features=4,
        num_extra_features=num_features,
        hidden_channels=params["hidden_channels"]
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["lr"]
    )

    best_r2 = float("-inf")
    best_model = None
    best_optimizer = None
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        patience=3,
        factor=0.5,
    )

    with mf.start_run(run_name):

        for epoch in range(params["epochs"]):

            loss = train_hybrid_model(
                model,
                train_loader,
                optimizer,
                device,
                epoch
            )

            train_r2, train_mae, _, _ = evaluate_hybrid_model(
                model,
                train_loader,
                device
            )

            mlflow.log_metric("train_r2", train_r2, step=epoch)
            mlflow.log_metric("train_mae", train_mae, step=epoch)

            r2, mae, _, _ = evaluate_hybrid_model(
                model,
                test_loader,
                device
            )
            scheduler.step(r2)

            mlflow.log_metric("train_mse", loss, step=epoch)
            mlflow.log_metric("val_r2", r2, step=epoch)
            mlflow.log_metric("val_mae", mae, step=epoch)

            if r2 > best_r2:
                best_r2 = r2
                best_model = copy.deepcopy(model)
                best_optimizer = copy.deepcopy(optimizer)

        return best_r2, best_model, best_optimizer
    
def process_row(row):
    scaled_features = scaler.transform(row[features].values.reshape(1, -1)).flatten()
    return mol_to_graph(row['canonical_smiles'], row['pic50'], scaled_features)

seed_everything(42)

features = [
    'alogp', 'psa', 'hba', 'hbd', 'num_ro5_violations', 'qed_weighted',
    'logP_over_PSA', 'HBA_HBD_sum'
]
num_features = len(features)
parquet_path = "parquets/subset_CHEMBL203_stratified.parquet"
df = pd.read_parquet(parquet_path)
dataset_path = "/home/pkuszn/repos/WSzI/src/notebooks/data/chembl_dataset.pt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(train_df[features])

print("Processing Train...")
train_data = Parallel(n_jobs=-1)(delayed(process_row)(row) for _, row in train_df.iterrows())
train_data = [d for d in train_data if d is not None]

print("Processing Test...")
test_data = Parallel(n_jobs=-1)(delayed(process_row)(row) for _, row in test_df.iterrows())
test_data = [d for d in test_data if d is not None]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

batch = next(iter(train_loader))

param_grid = [
    {"lr": 1e-2, "hidden_channels": 32, "epochs": 50},
    {"lr": 5e-3, "hidden_channels": 64, "epochs": 50},
    {"lr": 1e-3, "hidden_channels": 64, "epochs": 100},
    {"lr": 1e-3, "hidden_channels": 128, "epochs": 100},
    {"lr": 5e-4, "hidden_channels": 128, "epochs": 100},
]

mf = MLFlowManager(experiment_name="ChEMBL_HybridModel_Scaffold_Split")

now = str(int(datetime.now().timestamp()))
run_name = f"Hybrid_Run_{now}"
print("Starting Training...")

best_config = None
best_score = float("-inf")
best_model = None
best_optimizer = None
best_runname = None
for i, params in enumerate(param_grid):
    gc.collect()
    torch.cuda.empty_cache()
    run_name = f"Hybrid_tune_{i}_{int(datetime.now().timestamp())}"

    print(f"\nRunning config {i+1}/{len(param_grid)}: {params}")

    score, model, optimizer = train_one_run(
        params,
        train_loader,
        test_loader,
        device,
        run_name
    )

    if score > best_score:
        best_score = score
        best_config = params
        best_model = model
        best_optimizer = optimizer
        best_runname = run_name
    gc.collect()
    torch.cuda.empty_cache()

mlflow.log_metrics({"best_r2": best_score})
print("\nBEST CONFIG:", best_config)
print("BEST R2:", best_score)
for k, v in best_config.items(): # type: ignore
    mlflow.log_param(f"best_{k}", v)

model_save_path = f"model_{best_runname}_weights.pth"
checkpoint = {
    "model_state_dict": best_model.state_dict(), # type: ignore
    "optimizer_state_dict": best_optimizer.state_dict(), # type: ignore
    "params": best_config,
    "timestamp": datetime.now().isoformat()
}
torch.save(checkpoint, model_save_path)
print(f"Saved weight to {model_save_path}")
mlflow.log_artifact(model_save_path)

del optimizer
del model

Processing Train...


/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, b

Processing Test...


/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, b

Starting Training...

Running config 1/5: {'lr': 0.01, 'hidden_channels': 32, 'epochs': 50}
Epoch 0 | MSE: 3.9900 | MAE: 1.5395 | R2: -0.8704
Epoch 1 | MSE: 2.5944 | MAE: 1.3004 | R2: -0.2162
Epoch 2 | MSE: 2.3055 | MAE: 1.2316 | R2: -0.0808
Epoch 3 | MSE: 2.1667 | MAE: 1.1925 | R2: -0.0157
Epoch 4 | MSE: 2.0642 | MAE: 1.1668 | R2: 0.0324
Epoch 5 | MSE: 1.9579 | MAE: 1.1366 | R2: 0.0822
Epoch 6 | MSE: 1.9021 | MAE: 1.1216 | R2: 0.1083
Epoch 7 | MSE: 1.7766 | MAE: 1.0909 | R2: 0.1672
Epoch 8 | MSE: 1.6948 | MAE: 1.0572 | R2: 0.2055
Epoch 9 | MSE: 1.6401 | MAE: 1.0417 | R2: 0.2312
Epoch 10 | MSE: 1.5972 | MAE: 1.0235 | R2: 0.2513
Epoch 11 | MSE: 1.5711 | MAE: 1.0178 | R2: 0.2635
Epoch 12 | MSE: 1.5203 | MAE: 0.9985 | R2: 0.2873
Epoch 13 | MSE: 1.4509 | MAE: 0.9678 | R2: 0.3199
Epoch 14 | MSE: 1.4228 | MAE: 0.9635 | R2: 0.3330
Epoch 15 | MSE: 1.4216 | MAE: 0.9610 | R2: 0.3336
Epoch 16 | MSE: 1.4011 | MAE: 0.9543 | R2: 0.3432
Epoch 17 | MSE: 1.3985 | MAE: 0.9524 | R2: 0.3444
Epoch 18 | MSE